# Ali-CCP MTL Leaderboard (Colab)

Runs `run_experiments.py`: trains and evaluates all 13 models in this folder (3 classic
single-task baselines from `single_task_models.py` + 10 ESMM-family MTL architectures
from `models.py`) on the full Ali-CCP dataset via the shared harness in `harness.py`,
one epoch each at batch_size=4096 / embed_dim=18 / seed=42 (matched to the published
single-protocol leaderboard), and writes `results/aliccp_leaderboard.{json,md}`.

All model/training/eval logic lives in the `.py` files in this folder — this notebook
is just the Colab bootstrap (Drive mount + repo clone + deps) followed by one call into
`run_experiments.main()`. Depends on `datasets/aliccp/` for
parsed/normalized Parquet, filtered sparse vocabs, and sparse cardinalities.

In [ ]:
import os
if os.path.ismount('/content/drive'):
    print('Drive already mounted.')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print('Skipping drive mount (not in Colab UI or drive unavailable).')

In [ ]:
import os
WORK_DIR = '/content/drive/MyDrive/colab/repos'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os, subprocess, shutil
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

SKIP_GIT_REPO_SYNC = True
if os.environ.get('FORCE_GIT_SYNC', '').strip().lower() in ('1', 'true', 'yes'):
    SKIP_GIT_REPO_SYNC = False

if IN_COLAB:
    git_marker = os.path.join(repo_dir, '.git')
    if SKIP_GIT_REPO_SYNC:
        if os.path.isdir(repo_dir):
            os.chdir(repo_dir)
            print('[SKIP_GIT_REPO_SYNC] Using Drive copy.')
        else:
            subprocess.run(['git', 'clone', repo_url], check=True)
            os.chdir(repo_dir)
            subprocess.run(['git', 'checkout', branch_name], check=False)
    elif os.path.isdir(repo_dir) and os.path.isdir(git_marker):
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)
        subprocess.run(['git', 'reset', '--hard', f'origin/{branch_name}'], check=False)
    else:
        if os.path.exists(repo_dir):
            shutil.rmtree(repo_dir)
        subprocess.run(['git', 'clone', repo_url], check=True)
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)

In [ ]:
import subprocess
# Minimal deps actually imported by this folder's .py files (+ datasets/aliccp/):
# torch/pandas/numpy/pyarrow for data + tensors, scikit-learn for eval metrics.
subprocess.run(['pip', 'install', '-q', 'torch', 'pandas', 'numpy', 'scikit-learn', 'pyarrow'], check=True)

In [ ]:
import os, sys

os.chdir('aliccp_mtl_experiments')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import run_experiments
results = run_experiments.main()